# 商品明细阶段整理：从原始 CSV 复现经营指标

本 Notebook 调用公共函数，复现第四份 Notebook 的清洗、汇总与关联过程。你此前的手写记录保留原样。

重点理解输入和输出的粒度，以及为什么同时保留明细、订单汇总和覆盖检查。

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'
if str(project_dir / 'src') not in sys.path:
    sys.path.insert(0, str(project_dir / 'src'))
import items_cleaning
ic = importlib.reload(items_cleaning)

def read_raw(filename, usecols=None):
    return pd.read_csv(
        data_dir / filename, usecols=usecols, dtype=str,
        encoding='utf-8-sig', keep_default_na=False, na_values=['']
    )

items_raw = read_raw('olist_order_items_dataset.csv')
orders_raw = read_raw('olist_orders_dataset.csv')
product_keys = read_raw('olist_products_dataset.csv', ['product_id'])['product_id']
seller_keys = read_raw('olist_sellers_dataset.csv', ['seller_id'])['seller_id']

In [2]:
items_clean = ic.clean_order_items(items_raw)
reference_check = ic.check_item_references(items_clean, product_keys, seller_keys)
if reference_check.ne(0).any().any():
    raise ValueError('商品或卖家关联键需要先核查')
reference_check

,商品关联,卖家关联
维表主键缺失数,0,0
维表主键重复数,0,0
明细外键缺失数,0,0
非空但未匹配明细数,0,0


In [3]:
order_item_summary = ic.aggregate_order_items(items_clean)
orders_with_items = ic.join_orders_items(orders_raw, order_item_summary)
reconcile = ic.reconcile_item_totals(items_clean, order_item_summary, orders_with_items)
metrics = ic.summarize_item_metrics(items_clean, order_item_summary, orders_with_items)

print('明细行数：', len(items_clean))
print('订单汇总行数：', len(order_item_summary))
print('左关联后订单行数：', len(orders_with_items))
reconcile

明细行数： 112650
订单汇总行数： 98666
左关联后订单行数： 99441


,明细层,订单汇总层,关联后,汇总差额,关联差额
商品明细数量,112650,112650,112650,0,0
商品金额（分）,1359164370,1359164370,1359164370,0,0
运费（分）,225190954,225190954,225190954,0,0


In [4]:
print('零运费明细数：', metrics['zero_freight_item_rows'])
print('零运费订单数：', metrics['zero_freight_orders'])
print('优先核查订单数：', metrics['priority_missing_items'])
orders_with_items.loc[
    orders_with_items['flag_missing_items'], 'order_status'
].value_counts()

零运费明细数： 383
零运费订单数： 338
优先核查订单数： 3


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [5]:
print('已交付订单数：', metrics['delivered_orders'])
print('已交付商品金额：{:,.2f} BRL'.format(metrics['delivered_goods_cents'] / 100))
print('已交付运费：{:,.2f} BRL'.format(metrics['delivered_freight_cents'] / 100))
print('商品客单价：{:.2f} BRL'.format(metrics['delivered_goods_aov_brl']))

已交付订单数： 96478
已交付商品金额：13,221,498.11 BRL
已交付运费：2,198,275.64 BRL
商品客单价：137.04 BRL


## 保存与阶段边界

从项目目录运行 `python .\src\03_clean_items.py` 可重新生成 `data/processed/items_v1/` 和 `reports/items_v1/` 的输出。

补充类型规范：明细序号转换为正整数，并在转换后复核组合主键。原始字符串中的金额精确换算为整数分，零运费保留。

发货截止时间中存在 2020 年记录，当前保留原值，业务合理性仍待结合下单时间检查。GMV 是全量历史已交付商品金额基线，不含运费；后续仍需支付对账、正式分析窗口和敏感性分析。

完整说明见 `docs/04_商品明细阶段整理.md`。

## 我的理解

1. 为什么要同时保留商品明细表和订单汇总表？它们各适合分析什么？

我的回答：商品明细表保留商品和卖家等详细信息，适合商品、品类及卖家分析，也方便追溯和核对。订单汇总表把金额统一到“一行一个订单”的粒度，适合计算订单指标，以及关联订单状态、订单级支付金额。两张表服务于不同分析需求，因此都需要保留。

2. 为什么不能直接把商品明细和支付明细按订单 ID 关联后求和？

我的回答：同一订单可能包含多条商品明细和多条支付记录。直接按订单 ID 关联，会产生多对多匹配，使商品金额和支付金额重复出现，导致求和被放大。应先分别按订单汇总商品金额、运费和支付金额，再将两张订单级汇总表一对一关联。

下一课：支付表的主键、多条支付、订单级支付汇总和金额核对。